In [0]:
%skip
dbutils.fs.ls('/Volumes/adb_ws_wecare/landing/landing_volume/neon_postgres/hospitals/2026-10-22_08:50:41/')

In [0]:
# Create widgets
dbutils.widgets.text("source_system","")#neon_postgres
dbutils.widgets.text("table_name", "")#hospitals
dbutils.widgets.text("landing_timestamp", "")#2026-10-22_08:50:41
dbutils.widgets.text("source_file_format", "")#parquet


In [0]:

# Get widget values into variables
source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

print("=" * 80)
print("STEP 1: Reading job parameters")
print("=" * 80)
print(f"  source_system     : '{source_system}'")
print(f"  table_name        : '{table_name}'")
print(f"  landing_timestamp : '{landing_timestamp}'")
print(f"  source_file_format: '{source_file_format}'")
for _name, _val in [("source_system", source_system), ("table_name", table_name), ("landing_timestamp", landing_timestamp), ("source_file_format", source_file_format)]:
    if not _val:
        print(f"  WARNING: {_name} is empty")

In [0]:
print(source_system)
print(table_name)
print(landing_timestamp)
print(source_file_format)

In [0]:
landing_path = f'/Volumes/adb_ws_wecare/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/'

print("=" * 80)
print("STEP 2: Reading parquet files from landing volume")
print("=" * 80)
print(f"  Landing path : {landing_path}")

# Check whether the landing path (and its parent directories) exist
def path_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception as e:
        if 'FileNotFoundException' in str(e) or 'does not exist' in str(e).lower() or 'NOT_FOUND' in str(e):
            return False
        raise

if not path_exists(landing_path):
    print(f"  No new files received: path does not exist -> {landing_path}")
    print("  Exiting notebook without failure.")
    dbutils.notebook.exit("No new files received")

landing_files = [f for f in dbutils.fs.ls(landing_path) if f.name.endswith('.parquet')]
print(f"  Parquet files found: {len(landing_files)}")
for f in landing_files:
    print(f"    - {f.name} ({f.size} bytes)")
print("  Equivalent SQL query:")
print(f"    SELECT *\n    FROM parquet.`{landing_path}`")

if source_file_format == 'csv':
    landing_df = spark.read.format('csv') \
        .option('header', 'true') \
        .option('inferSchema', 'true') \
        .load(landing_path)
else:
    landing_df = spark.read.format('parquet') \
        .load(landing_path)

landing_count = landing_df.count()
print(f"  Columns ({len(landing_df.columns)}): {landing_df.columns}")
print(f"  Records read from landing: {landing_count}")
if landing_count == 0:
    print("  WARNING: No records found in the landing files")

In [0]:
from pyspark.sql.functions import lit, current_timestamp

updated_df = landing_df.withColumn("landing_timestamp", lit(landing_timestamp)) \
                       .withColumn("insert_timestamp", current_timestamp())

print("=" * 80)
print("STEP 3: Adding audit columns")
print("=" * 80)
print(f"  Added columns: landing_timestamp = '{landing_timestamp}', insert_timestamp = current_timestamp()")
print("  Equivalent SQL query:")
print(f"    SELECT *, '{landing_timestamp}' AS landing_timestamp, current_timestamp() AS insert_timestamp\n    FROM parquet.`{landing_path}`")
print(f"  Final columns ({len(updated_df.columns)}): {updated_df.columns}")

In [0]:
target_table = f"adb_ws_wecare.bronze.{table_name}"
target_exists = spark.catalog.tableExists(target_table)

print("=" * 80)
print("STEP 4: Writing data to bronze layer")
print("=" * 80)
print(f"  Target table        : {target_table}")
print(f"  Target table exists : {target_exists}" + ("" if target_exists else " (table will be created)"))
rows_before = spark.table(target_table).count() if target_exists else 0
print(f"  Rows in target before write: {rows_before}")
col_list = ', '.join([f'`{c}`' for c in updated_df.columns])
print("  Mode: APPEND")
print("  Equivalent SQL query:")
print(f"    INSERT INTO {target_table} ({col_list})\n    SELECT *, '{landing_timestamp}' AS landing_timestamp, current_timestamp() AS insert_timestamp\n    FROM parquet.`{landing_path}`")

updated_df.write.mode("append").saveAsTable(target_table)

rows_after = spark.table(target_table).count()
print(f"  Rows in target after write : {rows_after}")
print(f"  Rows appended              : {rows_after - rows_before}")
print(f"  APPEND load completed for {target_table}")

In [0]:
%skip
%sql
create schema adb_ws_wecare.bronze;
create schema adb_ws_wecare.silver;
create schema adb_ws_wecare.gold;

In [0]:
%skip
%sql
select * from adb_ws_wecare.bronze.hospitals